# 🤖 GitHub Copilot — Agent Mode, Instructions & the CLI

This notebook is a companion to the Airbnb data analysis workshop. It covers three features that take Copilot from a code autocomplete tool to a genuine working assistant:

| Feature | What it does |
|---|---|
| **Custom Instructions** | Tell Copilot about your project once — it remembers for every conversation |
| **Agent Mode** | Let Copilot plan and execute multi-step tasks across files and the terminal |
| **Copilot CLI** | Use Copilot from the command line to explain or generate shell commands |

---

**Prerequisites:** VS Code with the GitHub Copilot extension installed and signed in.

---
# Part 1 — Custom Instructions (`copilot-instructions.md`) 📋

## What is it?

A `copilot-instructions.md` file is a plain text file you place in your project at:

```
your-project/
└── .github/
    └── copilot-instructions.md   ← Copilot reads this automatically
```

GitHub Copilot reads this file automatically whenever you open the project. Everything in it is injected into every Copilot Chat conversation as background context — you never have to repeat yourself.

## Why does it matter?

Without custom instructions, Copilot has no idea:
- Which SQL dialect you're using
- What columns your DataFrame has
- What libraries and style conventions your team uses

**With** custom instructions, you can stop explaining the same things in every prompt. Instead of:

> *"We're using DuckDB, not regular SQL. The DataFrame is called `df`. It has columns: city, room_type, price... Now write a query to..."*

You just write:

> *"Write a query to find the top 5 neighbourhoods by average reviews."*

Copilot already knows the rest.

## What can you put in it?

Anything that helps Copilot give better answers for your specific project:

| Category | Examples |
|---|---|
| **Tools & libraries** | "Always use DuckDB for SQL", "Use plotly for interactive charts" |
| **Data context** | Column names, data types, what values mean |
| **Coding conventions** | "Add type hints", "Prefer f-strings", "No inline comments" |
| **Business context** | "This is a property market analysis", "Audience is non-technical" |
| **Constraints** | "Don't use deprecated pandas methods", "Keep functions under 30 lines" |

## The instructions file for this workshop

We've already created a `copilot-instructions.md` for the Airbnb workshop. Here's what it contains and why:

In [ ]:
# Read and display the instructions file
from pathlib import Path

instructions_path = Path(".github/copilot-instructions.md")

if instructions_path.exists():
    print(instructions_path.read_text())
else:
    print("File not found. Make sure you're running this notebook from the workshop folder.")

## Exercise 1 — See the difference custom instructions make

**Step 1:** Open Copilot Chat (`Cmd+Option+I` / `Ctrl+Alt+I`)

**Step 2:** Paste this prompt — notice Copilot uses DuckDB automatically, without you asking:

> 💬 *"Write a query to find the average availability_365 for each room type."*

Without the instructions file, Copilot would likely return generic SQL or pandas code. With it, you should see DuckDB syntax and the correct `.df()` call at the end.

**Step 3:** Try asking for a chart — notice it picks the right library:

> 💬 *"Create a chart showing the median number_of_reviews per city."*

---

## Exercise 2 — Customise the instructions for your own team

Open `.github/copilot-instructions.md` and try adding a section like:

```markdown
## Output style
- When asked to summarise findings, always write in bullet points
- Use £ symbols for price values, not $
- Avoid jargon — write for a finance analyst, not a data scientist
```

Then test it:

> 💬 *"Summarise what we know about Airbnb pricing in Amsterdam."*

---

## Tips for writing good instructions

- **Be specific, not vague.** "Use DuckDB" is better than "use good SQL practices."
- **Include examples** — a short code snippet is worth ten sentences of description.
- **Keep it focused.** Don't paste your entire schema — just the columns Copilot will actually need.
- **Update it when the project changes.** If you rename a column or switch libraries, update the file.
- **Share it with your team** — commit it to the repo so everyone gets the same Copilot behaviour.

---
# Part 2 — Copilot Agent Mode 🧠

## What is Agent Mode?

Standard Copilot Chat answers questions and generates code, but **you** still have to copy the code, paste it, run it, and decide what to do next.

**Agent Mode** is different. You give Copilot a goal, and it:

1. **Plans** the steps needed to achieve it
2. **Edits** files directly (you approve or reject each change)
3. **Runs terminal commands** to install packages, run scripts, check outputs
4. **Reads the results** and decides what to do next
5. **Keeps going** until the task is done (or asks you when it's unsure)

Think of it as the difference between asking a colleague *"how would I do X?"* versus asking them to *"go and do X."*

## How to switch to Agent Mode

1. Open Copilot Chat (`Cmd+Option+I` / `Ctrl+Alt+I`)
2. Click the dropdown at the top of the Chat panel that says **"Ask"**
3. Select **"Agent"**

You'll see the panel change — Agent mode shows a list of available tools Copilot can use (file editing, terminal, search, etc.).

## What can Agent Mode do?

| Capability | Example |
|---|---|
| Edit multiple files | *"Add error handling to every function in this notebook"* |
| Create new files | *"Create a new notebook that loads only the London data"* |
| Run terminal commands | *"Install the missing package and re-run this cell"* |
| Read error output | Copilot sees the traceback and fixes the cause automatically |
| Iterate until done | Copilot re-runs and adjusts until the output matches your goal |

## Exercise 3 — Try Agent Mode on a multi-step task

**Switch to Agent mode** (dropdown at the top of Chat → Agent), then try these prompts.
Watch how Copilot plans, edits, and runs — rather than just answering.

---

### Prompt A — Create a new analysis file

> 💬 *"Create a new Python script called `london_analysis.py` that: loads the Airbnb London data (2,000 row sample), calculates the top 10 neighbourhoods by median price using DuckDB, and saves the result as a CSV called `london_top_neighbourhoods.csv`."*

**What to notice:**
- Copilot creates the file without you copying and pasting
- It runs the script in the terminal and shows you the output
- If there's an error, it reads the traceback and fixes it

---

### Prompt B — Refactor across a whole notebook

> 💬 *"In air-bnb-workshop.ipynb, find every place where we use a hardcoded city name like 'London' or 'Paris' and replace it with a variable called `FOCUS_CITY` defined at the top of the notebook. Set the default to 'London'."*

**What to notice:**
- Agent reads the notebook, identifies every relevant cell, and proposes edits across all of them
- You review and accept/reject each change before it's applied

---

### Prompt C — Debug and fix an error

Run the cell below to introduce a deliberate bug, then ask Agent mode to fix it:

> 💬 *"The cell below is throwing an error. Fix it."*

In [ ]:
# This cell contains a deliberate bug — see if Agent mode can find and fix it.
# Run it first, then paste the error into Copilot Agent and ask it to fix the code.

import duckdb
import pandas as pd

# Simulate a small DataFrame
sample_data = pd.DataFrame({
    "city":           ["London", "London", "Paris", "Paris", "Amsterdam"],
    "room_type":      ["Entire home/apt", "Private room", "Entire home/apt", "Private room", "Entire home/apt"],
    "price":          [120, 65, 95, 50, 180],
    "number_of_reviews": [42, 18, 33, 7, 91],
    "availability_365":  [200, 150, 120, 80, 300],
})

# Bug: wrong column name used in the query
result = duckdb.query("""
    SELECT city, ROUND(MEDIAN(nightly_rate), 2) AS median_price
    FROM sample_data
    GROUP BY city
    ORDER BY median_price DESC
""").df()

result

## Tips for getting the best out of Agent Mode

| Do | Avoid |
|---|---|
| Describe the **goal**, not the steps | Don't micromanage — let Agent plan the approach |
| Mention the **output** you expect | Avoid vague goals like "make the code better" |
| Tell it which **files** to work with | Don't give it the entire codebase without narrowing the scope |
| **Review each change** before accepting | Don't auto-accept everything without reading it |
| Ask it to **explain its reasoning** if unsure | *"Why did you choose this approach?"* |

### When Agent Mode is most useful

- **Scaffolding new files** — creating a full script or notebook from a description
- **Refactoring** — renaming, restructuring, or adding error handling across multiple files
- **Debugging** — when you have an error and aren't sure where to start
- **Automating repetitive tasks** — "do this for each city" type tasks
- **End-to-end data pipelines** — load → clean → analyse → export, all in one prompt

---
# Part 3 — Copilot in the Terminal (CLI) 💻

## What is the GitHub Copilot CLI?

The Copilot CLI brings Copilot into your terminal. It's useful when you're:
- Running scripts and hitting unfamiliar shell errors
- Trying to remember the exact syntax for a command you use occasionally
- Working outside VS Code (SSH sessions, servers, CI pipelines)

## Installation

You need the GitHub CLI (`gh`) installed and authenticated first, then install the Copilot extension:

```bash
# Step 1: Install GitHub CLI (if not already installed)
# Mac:
brew install gh

# Step 2: Authenticate
gh auth login

# Step 3: Install the Copilot CLI extension
gh extension install github/gh-copilot

# Step 4: Verify it works
gh copilot --help
```

## The two main commands

| Command | What it does |
|---|---|
| `gh copilot explain` | Explain what a shell command does in plain English |
| `gh copilot suggest` | Suggest a shell command for a given task |

In [ ]:
# Check if the Copilot CLI is installed
import subprocess

result = subprocess.run(["gh", "copilot", "--version"], capture_output=True, text=True)

if result.returncode == 0:
    print("Copilot CLI is installed:", result.stdout.strip())
else:
    print("Copilot CLI not found.")
    print("Install with: gh extension install github/gh-copilot")
    print("(You'll need the GitHub CLI installed first: https://cli.github.com)")

## `gh copilot explain` — understand any command

Paste a command you've seen but don't fully understand, and Copilot explains it line by line.

### Example — run this in your terminal:

```bash
gh copilot explain "find . -name '*.csv' -newer requirements.txt -exec ls -lh {} ;"
```

**Expected output:**
Copilot will explain that this command finds all `.csv` files newer than `requirements.txt` and lists their sizes in human-readable format.

### More examples to try in your terminal:

```bash
gh copilot explain "jupyter nbconvert --to html air-bnb-workshop.ipynb"
```

```bash
gh copilot explain "pip freeze | grep -v '@ file' > requirements.txt"
```

```bash
gh copilot explain "ps aux | sort -rk 3,3 | head -5"
```

## `gh copilot suggest` — generate commands from plain English

Describe what you want to do, and Copilot suggests the right shell command.

### Example — run this in your terminal:

```bash
gh copilot suggest "convert the air-bnb-workshop.ipynb notebook to an HTML file"
```

Copilot will suggest something like:
```bash
jupyter nbconvert --to html air-bnb-workshop.ipynb
```
...and ask if you want to run it, copy it, or revise the request.

### More prompts to try:

```bash
gh copilot suggest "find all Python files modified in the last 7 days"
```

```bash
gh copilot suggest "show me how much disk space each folder in my current directory is using"
```

```bash
gh copilot suggest "create a zip file of all .ipynb files in the current folder"
```

---

## Useful aliases (optional setup)

If you use the CLI often, add these to your shell profile (`~/.zshrc` or `~/.bashrc`):

```bash
# Shorter aliases for Copilot CLI
alias '??'='gh copilot suggest'
alias 'explain'='gh copilot explain'
```

After reloading your shell, you can just type:
```bash
?? "zip all notebooks in this folder"
```

---
# Part 4 — Copilot in the VS Code Terminal 🖥️

You don't need to install the CLI extension to get Copilot help in the VS Code terminal. The **`@terminal`** Copilot Chat participant is built into VS Code.

## How to use `@terminal`

Open Copilot Chat (`Cmd+Option+I`), then prefix your question with `@terminal`:

| Prompt | What happens |
|---|---|
| `@terminal how do I find all .csv files in this folder?` | Gets a shell command with explanation |
| `@terminal explain this error: [paste error]` | Explains the terminal error and suggests a fix |
| `@terminal what command converts a notebook to a PDF?` | Suggests the right `nbconvert` command |

## Explain a terminal error

One of the most useful patterns: when a command fails, paste the error into `@terminal` and ask what went wrong.

**Example:** If `pip install` fails with a long error, try:

> `@terminal explain this error: ERROR: Could not find a version that satisfies the requirement duckdb==99.0`

Copilot will explain the error and suggest the correct command to find the available versions.

---
# Part 5 — Putting It All Together 🎯

Here's how all three features work together in a realistic workflow:

---

## Scenario: Building a monthly Airbnb market report

**Situation:** Your manager asks for a monthly report showing median price and booking demand by city. You need to automate it so it runs every month without manual work.

**Step 1 — Set up custom instructions** (do this once)

Add to `.github/copilot-instructions.md`:
```markdown
## Reporting conventions
- All reports should export to both CSV (for data teams) and HTML (for stakeholders)
- Use the city colour palette: London=#1f77b4, Paris=#ff7f0e, Amsterdam=#2ca02c
- Date format: YYYY-MM-DD
```

**Step 2 — Use Agent Mode to scaffold the report script**

> 💬 *"Create a Python script called `monthly_report.py` that: downloads 2,000 Airbnb listings per city, calculates median price and median reviews per city using DuckDB, generates a bar chart, and saves both a CSV summary and an HTML chart to a `reports/` folder with today's date in the filename."*

**Step 3 — Use the CLI to schedule it**

```bash
gh copilot suggest "run monthly_report.py every first Monday of the month at 9am using cron"
```

---

## Exercise 4 — Full workflow challenge

Using everything you've learned, try this end-to-end task:

1. **Update** `copilot-instructions.md` to add a reporting conventions section for your team
2. **Switch to Agent Mode** and ask it to create a `city_report.py` script that generates a summary for a single city (passed as a command-line argument)
3. **Run it** from the terminal: `python city_report.py London`
4. **Use the CLI** to find a command that converts the output CSV to a nicely formatted table in the terminal

> 💬 Suggested Agent prompt: *"Create city_report.py — a script that accepts a city name as a command-line argument, loads 1,000 Airbnb listings for that city, calculates: median price (if available), median reviews, top 5 neighbourhoods by listing count, and room type breakdown, then prints a formatted summary table to the console."*

---
# Summary & Quick Reference 📖

## Custom Instructions (`copilot-instructions.md`)

| | |
|---|---|
| **File location** | `.github/copilot-instructions.md` in your project root |
| **When it loads** | Automatically, on every Copilot Chat conversation in this project |
| **Best for** | Tools/libraries, data schemas, coding conventions, team style guides |
| **Shared?** | Yes — commit it to the repo so the whole team benefits |

---

## Agent Mode

| | |
|---|---|
| **How to enable** | Copilot Chat dropdown → switch from "Ask" to "Agent" |
| **Best for** | Multi-step tasks, creating files, debugging errors, refactoring |
| **Key tip** | Describe the *goal*, not the steps. Review each change before accepting. |
| **Difference from Chat** | Agent acts; Chat advises |

---

## Copilot CLI

| Command | Use |
|---|---|
| `gh copilot explain "<command>"` | Explain what a shell command does |
| `gh copilot suggest "<task>"` | Generate a shell command from plain English |
| `@terminal` in Copilot Chat | Same capabilities, directly in VS Code |

---

## When to use each feature

| Situation | Use |
|---|---|
| You keep explaining the same context to Copilot | Custom Instructions |
| You want Copilot to write and run code for you | Agent Mode |
| You need to create a new script or file from scratch | Agent Mode |
| You're debugging an error across multiple files | Agent Mode |
| You can't remember the right shell command | Copilot CLI (`suggest`) |
| You've copied a command and want to understand it | Copilot CLI (`explain`) |
| You're getting a terminal error you don't understand | `@terminal` in Chat |